# Fly neuPrint: Structural Paths Tutorial

This tutorial treats a connectome as a **structural** directed graph. We ask whether named sensory neurons can reach candidate descending neurons through synaptic paths, then measure how reachability changes after virtual removal of selected nodes. These are hypotheses about anatomical communication routes—not demonstrations of neural activity, causal circuit function, or whole-animal behavior.

The default analysis uses a small synthetic graph so that it runs without network access or a neuPrint account. An explicitly opt-in cell shows how to configure a neuPrint client for an authorized public dataset.

## Quick start

For the default offline path, run the minimal-install cell and then run the synthetic graph cells from top to bottom. You should see a directed toy network, a count of structurally reachable sensory-to-descending pairs, and a virtual-lesion sensitivity bar chart. Later cells are optional advanced previews for authorized neuPrint access; they do not provide data to, or change, the synthetic analysis.


## Prerequisites

- Python 3.9+ and basic directed-graph concepts
- Familiarity with synapses, cell types, and the difference between structural and functional connectivity
- For the optional live query: a neuPrint account and dataset-specific access token

## Setup

Run the following minimal installation for the default offline tutorial. No large connectome data are downloaded by default.

In [ ]:
%pip install -q numpy networkx matplotlib

### Optional installation for advanced live neuPrint previews

Install this package only if you plan to enable the advanced preview cells below. The default synthetic analysis does not import it.


In [ ]:
%pip install -q neuprint-python

In [ ]:
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
print('networkx', nx.__version__)

## Advanced preview (optional): configure a live neuPrint client

Set `USE_LIVE_NEUPRINT=True` only after exporting `NEUPRINT_APPLICATION_CREDENTIALS` (or providing a token below) and confirming the server and dataset you are authorized to use. Dataset names, available annotations, and query permissions vary. This cell deliberately does not run a broad whole-CNS query; inspect the dataset schema and select biologically justified cell classes before scaling up.

In [ ]:
USE_LIVE_NEUPRINT = False
NEUPRINT_SERVER = 'https://neuprint.janelia.org'
NEUPRINT_DATASET = None  # enter an accessible dataset name; this tutorial does not assume a CNS release

if USE_LIVE_NEUPRINT:
    from neuprint import Client
    token = os.environ.get('NEUPRINT_APPLICATION_CREDENTIALS')
    if not token:
        raise RuntimeError('Set NEUPRINT_APPLICATION_CREDENTIALS before enabling live access.')
    if not isinstance(NEUPRINT_DATASET, str) or not NEUPRINT_DATASET:
        raise RuntimeError('Set NEUPRINT_DATASET to an accessible dataset name before enabling live access.')
    client = Client(NEUPRINT_SERVER, dataset=NEUPRINT_DATASET, token=token)
    print(client.fetch_db_version())
else:
    print('Live neuPrint access is disabled; continuing with synthetic connectivity.')

## Advanced preview (optional): retrieve a small, constrained neuPrint subgraph

This advanced preview does not feed the synthetic analysis below. First run the configuration cell with `USE_LIVE_NEUPRINT=True`. Then replace the empty `LIVE_BODY_IDS` list with a small set of **actual body IDs** that you selected from the accessible dataset's explorer or a documented cell-type query. The same IDs are used as both sources and targets, so this fetches only neurons and connections within that selected set. Do not substitute names, skeleton IDs, or IDs from another dataset release.

In [ ]:
FETCH_LIVE_NEUPRINT = False
LIVE_BODY_IDS = []  # e.g., replace with verified integer body IDs from NEUPRINT_DATASET

if FETCH_LIVE_NEUPRINT:
    if not USE_LIVE_NEUPRINT or 'client' not in globals():
        raise RuntimeError('Enable and run the neuPrint configuration cell first.')
    if not LIVE_BODY_IDS or not all(isinstance(body_id, (int, np.integer)) for body_id in LIVE_BODY_IDS):
        raise ValueError('Provide a non-empty, verified list of integer body IDs for this dataset.')
    from neuprint import NeuronCriteria as NC, fetch_adjacencies, fetch_neurons

    selected = NC(bodyId=LIVE_BODY_IDS)
    live_neurons, live_roi_counts = fetch_neurons(selected, client=client)
    _, live_connections = fetch_adjacencies(
        selected, selected, min_total_weight=1, client=client
    )
    print(f'Retrieved {len(live_neurons)} selected neurons and {len(live_connections)} within-set connections.')
    display(live_neurons[['bodyId', 'type', 'instance']].head())
    display(live_connections.head())
else:
    print('Live subgraph retrieval is disabled; no neuPrint data will be downloaded.')

## A reproducible toy sensory-to-descending graph

Each directed edge stores an integer synthetic synapse count. The labels are illustrative and are not identities or measurements from a released fly connectome. Edge weights are used as a thresholding convenience here; synapse counts are not equivalent to efficacy.

In [ ]:
edges = [
    ('visual_1', 'optic_interneuron', 28), ('visual_2', 'optic_interneuron', 18),
    ('olfactory_1', 'lateral_horn', 25), ('mechanosensory_1', 'ventral_interneuron', 30),
    ('optic_interneuron', 'central_hub', 22), ('lateral_horn', 'central_hub', 19),
    ('ventral_interneuron', 'central_hub', 16), ('central_hub', 'premotor_1', 24),
    ('central_hub', 'premotor_2', 12), ('premotor_1', 'descending_A', 21),
    ('premotor_2', 'descending_B', 20), ('optic_interneuron', 'descending_B', 7)
]
G = nx.DiGraph()
G.add_weighted_edges_from(edges, weight='synapses')
sensory = ['visual_1', 'visual_2', 'olfactory_1', 'mechanosensory_1']
descending = ['descending_A', 'descending_B']
print(f'{G.number_of_nodes()} nodes; {G.number_of_edges()} directed edges')

pos = nx.spring_layout(G, seed=4)
nx.draw_networkx(G, pos, node_color=['tab:orange' if n in sensory else 'tab:green' if n in descending else 'lightgray' for n in G], arrows=True, font_size=8)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'synapses'), font_size=7)
plt.title('Synthetic directed synapse graph (edge label = count)')
plt.axis('off');


## Reachability under a synapse-count threshold

A directed path means only that a sequence of retained anatomical edges exists. Testing several thresholds is preferable to declaring a single threshold biologically decisive.

In [ ]:
def threshold_graph(graph, minimum_synapses):
    return nx.DiGraph((u, v, d) for u, v, d in graph.edges(data=True)
                      if d['synapses'] >= minimum_synapses)

def reachability_table(graph, sources, targets):
    return {(source, target): nx.has_path(graph, source, target)
            if source in graph and target in graph else False
            for source in sources for target in targets}

for threshold in (1, 10, 20):
    H = threshold_graph(G, threshold)
    table = reachability_table(H, sensory, descending)
    print(f'minimum synapses = {threshold}: {sum(table.values())}/{len(table)} reachable source-target pairs')

H = threshold_graph(G, 10)
for source in sensory:
    for target in descending:
        if nx.has_path(H, source, target):
            print(source, '->', target, ':', nx.shortest_path(H, source, target))

## Virtual lesioning as a graph sensitivity analysis

Removing a node in this model is a *virtual lesion*: it asks how graph reachability depends on that node. It does not predict the effect of an experimental lesion, which can involve compensation, state dependence, physiology, and connections absent from the graph.

In [ ]:
def reachable_pair_count(graph):
    return sum(reachability_table(graph, sensory, descending).values())

baseline = reachable_pair_count(H)
results = []
for candidate in sorted(set(H) - set(sensory) - set(descending)):
    lesioned = H.copy()
    lesioned.remove_node(candidate)
    remaining = reachable_pair_count(lesioned)
    results.append((candidate, baseline - remaining, remaining))

for node, lost, remaining in sorted(results, key=lambda row: -row[1]):
    print(f'{node:20s} removes {lost} reachable pairs; {remaining}/{baseline} remain')

plt.bar([r[0] for r in results], [r[1] for r in results], color='tab:red')
plt.ylabel('Reachable sensory-to-descending pairs lost')
plt.title('Virtual-lesion sensitivity at ≥10 synthetic synapses')
plt.xticks(rotation=30, ha='right'); plt.tight_layout()

## Interpreting and extending the analysis

For real data, document the dataset release, cell-selection criteria, ROI restrictions, synapse threshold, treatment of fragments, and whether indirect paths are biologically plausible. Compare against degree-preserving or cell-type-aware nulls when making enrichment claims. Pair this structural analysis with physiology and behavioral experiments before making functional or behavioral inferences.

## References

- Clements, J., Dolafi, T., Umayam, L., et al. (2020). neuPrint: Analysis tools for EM connectomics. *bioRxiv*. https://doi.org/10.1101/2020.01.16.909465
- Scheffer, L. K., Xu, C. S., Januszewski, M., et al. (2020). A connectome and analysis of the adult *Drosophila* central brain. *eLife*, 9, e57443. https://doi.org/10.7554/eLife.57443
- Dorkenwald, S., Matsliah, A., Sterling, A. R., et al. (2024). Neuronal wiring diagram of an adult brain. *Nature*, 634, 124–138. https://doi.org/10.1038/s41586-024-07558-y
- neuPrint documentation. https://connectome-neuprint.github.io/neuprint-python/docs/

## License

This notebook is released under the repository's license. The synthetic graph is created for teaching; neuPrint datasets and software remain subject to their respective access terms, licenses, and citation requirements.